# NLLB-200 Sinhala Subtitle Fine-tuning

**Before running:** `Runtime → Change runtime type → T4 GPU`

This notebook:
1. Checks GPU and mounts Google Drive
2. Installs dependencies
3. Copies project files from Drive
4. Runs a quick smoke test (100 pairs, 1 step)
5. Runs full training (3 epochs, ~7-8 hrs on T4)
6. Tests the final model with sample translations

Checkpoints save directly back to Drive so progress is never lost.

## Cell 1 — Check GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Go to Runtime -> Change runtime type -> T4 GPU, "
        "then reconnect and re-run."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU  : {gpu_name}")
print(f"VRAM : {vram_gb:.1f} GB")
print(f"CUDA : {torch.version.cuda}")

# Recommend batch size based on VRAM
if vram_gb >= 20:
    BATCH = 16
elif vram_gb >= 14:
    BATCH = 8
else:
    BATCH = 4

print(f"\nRecommended batch size for your GPU: {BATCH}")
print("(You can override this in Cell 5)")

## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── CONFIGURE THESE PATHS TO MATCH YOUR DRIVE FOLDER STRUCTURE ──
DRIVE_BASE    = "/content/drive/MyDrive/sinhala-subtitle"
SCRIPTS_DIR   = f"{DRIVE_BASE}/sinhala-finetune"       # train.py, dataset.py etc.
DATA_FILE     = f"{DRIVE_BASE}/clean_pairs_final.tsv"  # 522k clean pairs
CHECKPOINT_DIR = f"{DRIVE_BASE}/checkpoints"           # where models are saved
# ────────────────────────────────────────────────────────────────

# Verify paths exist
ok = True
for label, path in [("Scripts dir", SCRIPTS_DIR), ("Data file", DATA_FILE)]:
    exists = os.path.exists(path)
    status = "OK" if exists else "NOT FOUND"
    print(f"  {status}  {label}: {path}")
    if not exists:
        ok = False

if not ok:
    print("\nFix the paths above then re-run this cell.")
else:
    print("\nAll paths verified.")

## Cell 3 — Install dependencies

In [ ]:
%%capture install_log
!pip install \
    transformers==4.40.2 \
    datasets==2.19.0 \
    accelerate==0.29.3 \
    sentencepiece==0.2.0 \
    sacrebleu==2.4.0 \
    evaluate==0.4.1 \
    -q

print("Dependencies installed.")

# Verify key imports
import transformers, datasets, accelerate, sentencepiece, sacrebleu, evaluate
print(f"  transformers : {transformers.__version__}")
print(f"  datasets     : {datasets.__version__}")
print(f"  accelerate   : {accelerate.__version__}")

## Cell 4 — Copy project files into Colab runtime

In [ ]:
import shutil, os

# Copy scripts to /content/sinhala-finetune
LOCAL_SCRIPTS = "/content/sinhala-finetune"
if os.path.exists(LOCAL_SCRIPTS):
    shutil.rmtree(LOCAL_SCRIPTS)
shutil.copytree(SCRIPTS_DIR, LOCAL_SCRIPTS)
print(f"Scripts copied to {LOCAL_SCRIPTS}")

# Copy data to /content (faster local I/O during training)
LOCAL_DATA = "/content/clean_pairs_final.tsv"
if not os.path.exists(LOCAL_DATA):
    shutil.copy(DATA_FILE, LOCAL_DATA)
    print(f"Data copied to {LOCAL_DATA}")
else:
    print(f"Data already at {LOCAL_DATA}")

# Verify
import pandas as pd
df = pd.read_csv(LOCAL_DATA, sep='\t', dtype=str).dropna()
print(f"\nDataset loaded: {len(df):,} pairs")
print("\nFirst 3 rows:")
for _, row in df.head(3).iterrows():
    print(f"  EN: {row.iloc[1]}")
    print(f"  SI: {row.iloc[2]}")
    print()

## Cell 5 — Smoke test (100 pairs, 1 step — ~3 min)

Run this before the full training to catch any issues early.

In [ ]:
import subprocess, sys

smoke_cmd = [
    sys.executable, "/content/sinhala-finetune/train.py",
    "--data",       LOCAL_DATA,
    "--output-dir", "/content/smoke_test",
    "--epochs",     "1",
    "--batch-size", str(BATCH),
    "--grad-accum", "1",
    "--sample",     "100",      # only 100 pairs
    "--save-steps", "999999",   # don't save during smoke
    "--eval-steps", "999999",
    "--num-proc",   "1",
    "--fp16",
]

print("Running smoke test...")
print(" ".join(smoke_cmd))
print()

result = subprocess.run(smoke_cmd, cwd="/content/sinhala-finetune", capture_output=False)

if result.returncode == 0:
    print("\nSmoke test PASSED. Ready for full training.")
else:
    print(f"\nSmoke test FAILED (exit code {result.returncode}).")
    print("Check the output above for errors before running full training.")

## Cell 6 — Full training (~7-8 hrs on T4)

Checkpoints save to Drive every 500 steps. If the session disconnects,
re-run Cells 1-4 then use Cell 7 (resume) instead of this cell.

In [ ]:
import subprocess, sys, os

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

train_cmd = [
    sys.executable, "/content/sinhala-finetune/train.py",
    "--data",          LOCAL_DATA,
    "--output-dir",    CHECKPOINT_DIR,   # saves directly to Drive
    "--epochs",        "3",
    "--batch-size",    str(BATCH),
    "--grad-accum",    "4",
    "--lr",            "5e-5",
    "--warmup",        "500",
    "--val-size",      "2000",
    "--save-steps",    "500",
    "--eval-steps",    "500",
    "--logging-steps", "100",
    "--num-proc",      "1",
    "--fp16",
]

print("Starting full training...")
print("Checkpoints will save to:", CHECKPOINT_DIR)
print()
print(" ".join(train_cmd))
print()

result = subprocess.run(train_cmd, cwd="/content/sinhala-finetune")

if result.returncode == 0:
    print("\nTraining complete!")
else:
    print(f"\nTraining exited with code {result.returncode}.")

## Cell 7 — Resume training after disconnect (run instead of Cell 6)

In [ ]:
import subprocess, sys, os, glob

# Auto-detect the latest checkpoint run folder
run_dirs = sorted(glob.glob(f"{CHECKPOINT_DIR}/nllb-sinhala-english-*/"))
if not run_dirs:
    print("No checkpoint run folders found in:", CHECKPOINT_DIR)
    print("Run Cell 6 instead (fresh start).")
else:
    RESUME_DIR = run_dirs[-1].rstrip("/")
    print(f"Resuming from: {RESUME_DIR}")

    resume_cmd = [
        sys.executable, "/content/sinhala-finetune/train.py",
        "--data",          LOCAL_DATA,
        "--output-dir",    CHECKPOINT_DIR,
        "--epochs",        "3",
        "--batch-size",    str(BATCH),
        "--grad-accum",    "4",
        "--lr",            "5e-5",
        "--warmup",        "500",
        "--val-size",      "2000",
        "--save-steps",    "500",
        "--eval-steps",    "500",
        "--logging-steps", "100",
        "--num-proc",      "1",
        "--fp16",
        "--resume",        RESUME_DIR,
    ]

    result = subprocess.run(resume_cmd, cwd="/content/sinhala-finetune")
    print(f"\nExited with code {result.returncode}.")

## Cell 8 — Test the trained model with sample sentences

In [ ]:
import glob, os, sys
sys.path.insert(0, "/content/sinhala-finetune")

# Find the final model
finals = sorted(glob.glob(f"{CHECKPOINT_DIR}/nllb-sinhala-english-*/final/"))
if not finals:
    print("No final model found yet — training may still be running.")
else:
    MODEL_PATH = finals[-1].rstrip("/")
    print(f"Loading model from: {MODEL_PATH}")

    from transformers import AutoModelForSeq2SeqLM, NllbTokenizerFast
    import torch

    tokenizer = NllbTokenizerFast.from_pretrained(MODEL_PATH)
    model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)
    device    = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device).eval()
    print(f"Model loaded on {device}\n")

    def translate(text, num_beams=4):
        tokenizer.src_lang = "eng_Latn"
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(device)
        with torch.no_grad():
            ids = model.generate(
                **inputs,
                forced_bos_token_id=tokenizer.lang_code_to_id["sin_Sinh"],
                num_beams=num_beams,
                max_new_tokens=128,
                no_repeat_ngram_size=3,
            )
        return tokenizer.decode(ids[0], skip_special_tokens=True)

    # Test sentences — mix of simple, colloquial, and named entities
    test_sentences = [
        "I can't believe you did that.",
        "Let's get out of here before they come back.",
        "She never told me about the money.",
        "You have to trust me on this one.",
        "He works at General Electric in New York.",
        "I'll meet you at the station at half past seven.",
        "Don't worry, everything is going to be fine.",
        "Why didn't you call me when it happened?",
    ]

    print("=" * 60)
    print("  Sample translations")
    print("=" * 60)
    for sent in test_sentences:
        si = translate(sent)
        print(f"  EN: {sent}")
        print(f"  SI: {si}")
        print()

## Cell 9 — BLEU benchmark against validation set

In [ ]:
import sys, glob
sys.path.insert(0, "/content/sinhala-finetune")
import evaluate as ev
import pandas as pd

# Use 200 random pairs from the clean set as a quick benchmark
df = pd.read_csv(LOCAL_DATA, sep='\t', dtype=str).dropna()
sample = df.sample(200, random_state=99)
pairs  = [{"src": row.iloc[1], "ref": row.iloc[2]} for _, row in sample.iterrows()]

print("Running BLEU benchmark on 200 sample pairs...")
result = ev.benchmark_checkpoint(
    model, tokenizer, pairs,
    src_lang="eng_Latn",
    tgt_lang="sin_Sinh",
    device=device,
)

print(f"\n  BLEU : {result['bleu']}")
print(f"  chrF : {result['chrf']}")
print("\n  --- 5 sample predictions ---")
for s in result["samples"]:
    print(f"  SRC  : {s['src']}")
    print(f"  REF  : {s['ref']}")
    print(f"  PRED : {s['pred']}")
    print()

## Cell 10 — Translate a .srt file (optional)

In [ ]:
# Upload a .srt file to Colab first, then set the path below
from google.colab import files
uploaded = files.upload()   # triggers file picker

import os, sys
srt_filename = list(uploaded.keys())[0]
srt_path     = f"/content/{srt_filename}"
out_path     = srt_path.replace(".srt", "_sinhala.srt")

print(f"Translating: {srt_filename}")

result = os.popen(
    f"{sys.executable} /content/sinhala-finetune/translate.py "
    f"--model {MODEL_PATH} "
    f"--srt {srt_path} "
    f"--out {out_path}"
).read()
print(result)

# Download the translated file
files.download(out_path)
print(f"Download triggered: {out_path}")